# IDM Run 3 — analysis workflow walkthrough

A hands-on tour of the `idm` package: the coffea-2025, **named-config** analysis flow.
By the end you will know **what to edit, where** to add an object, a cut, or a histogram,
how to run the analysis, and how to make a plot.

**Prerequisites:** the `idm_venv` environment + the `IDM (LCG_107 Py3.11)` Jupyter kernel
(see [`SETUP_Run3.md`](../../SETUP_Run3.md)). Select that kernel for this notebook, and run
from the repo root.

**The big idea:** you define analysis content *once, by name* in `idm/definitions/`, and you
*select by name* when you run. A new study changes **configuration**, not analysis code.

## 0. Imports

In [ ]:
import os, sys
sys.path.insert(0, "python_analysis/analysisTools")  # for the ntuple schema (until it's packaged)

import awkward as ak
import mplhep as hep
import matplotlib.pyplot as plt
from coffea.nanoevents import NanoEventsFactory
from mySchema_newCoffea import MySchema

from idm.tools.processor import IdmProcessor
from idm.definitions.cuts import cut_defs
from idm.definitions.hists import hist_defs

hep.style.use("CMS")

## 1. Load an ntuple

The **schema** (`MySchema`) reads the flat ROOT ntuple (`TTree "ntuples/outT"`) and turns its
branches into physics-object collections you can use directly — `events.Electron`, `events.Muon`,
and so on.

In [ ]:
# Point this at a flat ntuple (a ROOT file with the TTree "ntuples/outT").
# On LPC the ntuples live on EOS; read them over xrootd, e.g.
#   root://cmseos.fnal.gov//store/group/lpcmetx/iDMe/<...>.root
ntuple_path = os.environ.get("IDM_TUTORIAL_NTUPLE", "REPLACE_ME_with_a_flat_ntuple.root")

events = NanoEventsFactory.from_root(
    {ntuple_path: "ntuples/outT"}, schemaclass=MySchema,
).events()

# the object collections the schema built:
print(f"{len(events)} events")
sorted(events.fields)

## 2. Look at the objects

Each collection is a jagged array (a variable-length list per event). You can read any field
and do columnar maths on it — no event loops.

In [ ]:
print("muons / event :", ak.to_list(ak.num(events.Muon, axis=1))[:10])
print("a few muon pT  :", [round(p, 1) for p in ak.to_list(ak.flatten(events.Muon.pt))[:8]], "GeV")

## 3. The named-config flow — *what to edit, where*

All analysis content lives in three files, as dictionaries of **named** callables:

| file | dict | each entry is |
|---|---|---|
| `idm/definitions/objects.py` | `obj_defs` | `name → f(events)` returning a collection |
| `idm/definitions/cuts.py`    | `cut_defs`  | `name → f(events)` returning a per-event mask |
| `idm/definitions/hists.py`   | `hist_defs` | `name → {"axis": <hist.axis>, "fill": f(events) → array}` |

**To add a histogram**, add one entry to `hist_defs`. **To add a cut**, add to `cut_defs`.
That is the whole edit — nothing in the processor changes.

In [ ]:
print("available cuts :", list(cut_defs))
print("available hists:", list(hist_defs))
hist_defs["muon_pt"]   # peek at one definition

## 4. Run the analysis — *select by name*

`IdmProcessor` AND-s the chosen cuts into an event selection, fills the chosen histograms, and
returns them with a cutflow. You choose *which* by name — the same engine runs any configuration.

In [ ]:
p = IdmProcessor(cuts=["has_muon"], hists=["muon_pt", "n_electron"])
out = p.process(events)
print("cutflow:", out["cutflow"])
out["hists"]["muon_pt"]

## 5. Plot it (CMS style)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
hep.histplot(out["hists"]["muon_pt"], ax=ax, histtype="step")
ax.set_xlabel(r"PF muon $p_{T}$ [GeV]")
ax.set_ylabel("muons")
hep.cms.label("Preliminary", data=False, com=13.6, ax=ax)
plt.show()

## 6. Scaling up & provenance (pointers)

- **Run distributed** over many files on LPC HTCondor: build a client with
  `idm.tools.scaleout.make_lpc_client()` and drive the processor with coffea's runner.
- **Record provenance** for an output: `idm.tools.metadata.write_run_metadata(...)` writes a
  `.meta.yaml` sidecar (fileset, git commit, coffea version, schema) next to your `.coffea`.

## Recap — what to edit, where

| To add a… | edit | then run with |
|---|---|---|
| object | `idm/definitions/objects.py` | (referenced by cuts/hists) |
| cut | `idm/definitions/cuts.py` | `IdmProcessor(cuts=[...])` |
| histogram | `idm/definitions/hists.py` | `IdmProcessor(hists=[...])` |

You only ever edit the **named definitions** — `idm/tools/processor.py` stays the same.